# 6. Comparative evaluation

Run all systems on the same artifact and answer with measurements: Did specialization improve defect recall enough to justify extra calls, tokens, latency, cost, and operational complexity? The single reviewer is allowed to win.

## Before you begin

**Choose one:** use the structured mock reviewer for deterministic classroom work, or OpenRouter when the issued key is available. Never send private code.

### Learning outcomes

Calculate recall, false positives, calls, tokens, latency, and cost on the same artifact.

Architecture reference: [Day 4 diagrams D15](../../diagrams/source/day_04.md).

### Expected observation

Offline orchestration yields 5/9, 6/9, and 9/9; live model results may vary and must be preserved.

## Concept briefing

## Evaluating nondeterministic systems

Do not assert exact model wording. Test invariants and outcomes:

- Is every finding structurally valid?
- Does evidence refer to the supplied artifact?
- How many known defects were found?
- How many unsupported findings were reported?
- How many duplicates survived synthesis?
- How many calls and tokens were used?
- Did the system terminate within its bounds?

One run is an anecdote. Repeat model experiments with the same model, prompt version,
temperature and artifact. Report variance rather than selecting the best result.

## Capability can change the architecture conclusion

A weaker instruction-following model may benefit disproportionately from narrow prompts.
A stronger model may handle the general review well enough that specialist calls add
little value. Therefore "multi-agent is better" may actually mean "decomposition
compensated for this model under this task and prompt."

An instructor may repeat the same golden-set experiment on a currently strong reference
model. The lesson is not brand ranking. It is that model capability, cost and reliability
are architecture inputs.

## Cost and latency arithmetic

Approximate run cost as:

```text
sum of input tokens across calls
+ sum of output and reasoning tokens
+ retries
```

If the same 1,000-token artifact is sent to three specialists, the input is paid three
times unless caching or a provider feature changes the calculation. Parallel execution
may reduce elapsed time while preserving or increasing total cost.

A fair comparison records recall, false positives, calls, tokens, latency, estimated cost
and debugging complexity. The chosen system should be the smallest one that meets the
quality requirement.


In [ ]:
from pathlib import Path
import sys, json
DAY=Path.cwd()
if (DAY/"day_04_multi_agent_systems").exists(): DAY=DAY/"day_04_multi_agent_systems"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"review_team").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
SOURCE=(DAY/"data"/"seeded_artifact"/"order_service.py").read_text(encoding="utf-8")
GOLDEN=DAY/"data"/"golden_defects.json"
print("Artifact lines:",len(SOURCE.splitlines()))

In [ ]:
import os
from review_team import *
provider=OpenRouterReviewer() if os.getenv("OPENROUTER_API_KEY") else MockStructuredReviewer()
runs=[run_model_review(SOURCE,provider,"general"),run_augmented(SOURCE),run_model_multi(SOURCE,provider)]
rows=[evaluate(run,GOLDEN,price_per_million_tokens=0.0) for run in runs]
headers=["system","found","recall","false_positives","duplicates","model_calls","estimated_tokens","elapsed_ms","estimated_cost_usd"]
print(" | ".join(headers))
for row in rows: print(" | ".join(str(row[h]) for h in headers))

## Interpret carefully

Our offline specialists encode known category patterns, so their 100% result validates orchestration—not general model intelligence. A live A/B should hide the golden set, repeat trials, pin model/configuration, and report variance. Token counts here approximate repeated source input; provider usage is preferable for live runs.

In [ ]:
for row in rows: print(row["system"],"missed:",row["missed"])
best=max(rows,key=lambda r:(r["recall"],-r["model_calls"]))
print("Best under recall-then-fewer-calls rule:",best["system"])

## Required live observation

Run one single-reviewer and one bounded specialist comparison with the issued model. Preserve raw structured results; use the captured comparison if the service is unavailable.


## Your turn

Hand-calculate recall for one run, then repeat a model A/B twice and report variance.

## Recap

Choose the smallest system that meets measured quality requirements.